# Indic Language Identification — TF / IDF Normalization Study
### Comparing 6 combinations of normalized vs. unnormalized TF and IDF, all with a from-scratch Logistic Regression classifier

This notebook reuses the **same dataset, sentence-tokenization, and word/char n-gram feature extraction** from the previous assignment (word unigrams + bigrams, character 2/3/4-grams). It extends the TF-IDF vectorizer with:

**Term Frequency (TF) — 3 variants**
1. **Unnormalized (raw) TF** — the raw n-gram count in the sentence.
2. **Normalization (a): length normalization** — tf(t,d) = count(t,d) / (total number of tokens in d). This is what Assignment 1's model already used.
3. **Normalization (b): max-frequency normalization** — tf(t,d) = count(t,d) / (count of the most frequent term in d), i.e. divide by the frequency of the most frequent term *within that sentence*. This bounds TF to [0,1] and prevents very long / repetitive sentences from dominating.

**Inverse Document Frequency (IDF) — 2 variants**
1. **Unnormalized (standard) IDF** — the same smoothed formula from Assignment 1: idf(t) = ln((1+N)/(1+df(t))) + 1.
2. **Normalized IDF (based on the most frequent term across the corpus)** — idf(t) = ln((1+max_df)/(1+df(t))), where max_df is the document frequency of the single most common n-gram **across the whole training set**. Every term's IDF is now expressed *relative to* the most frequent term: the most common term gets an IDF near 0, and rarer terms get progressively larger IDF, scaled by how dominant the corpus's most frequent term is.

**3 TF variants x 2 IDF variants = 6 combinations.** For each, we build a TF-IDF matrix, train an identical from-scratch multinomial logistic regression classifier, and compare macro-F1 on validation and test — all metrics computed without `sklearn`.


In [3]:
import re
import json
import math
import time
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [4]:
TRAIN_PATH = "/content/train.csv"
VAL_PATH   = "/content/validation.csv"
TEST_PATH  = "/content/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

for d in (train_df, val_df, test_df):
    d.dropna(subset=["text", "label"], inplace=True)
    d["text"] = d["text"].astype(str)

print("Train:", train_df.shape, " Validation:", val_df.shape, " Test:", test_df.shape)

languages = sorted(train_df["label"].unique())
label2idx = {lang: i for i, lang in enumerate(languages)}
idx2label = {i: lang for lang, i in label2idx.items()}
print(f"Number of languages: {len(languages)}")


Train: (18400, 2)  Validation: (2300, 2)  Test: (2300, 2)
Number of languages: 23


In [5]:
def word_tokenize(text):
    return re.findall(r"\S+", text.strip())


def word_ngrams(tokens, n):
    if len(tokens) < n:
        return []
    return [" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def char_ngrams(text, n):
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) < n:
        return []
    return [text[i:i + n] for i in range(len(text) - n + 1)]


def get_features(row):
    text = row["text"] if not isinstance(row, str) else row
    tokens = word_tokenize(text)
    features = []
    features += ["W1_" + f for f in word_ngrams(tokens, 1)]
    features += ["W2_" + f for f in word_ngrams(tokens, 2)]
    features += ["C2_" + f for f in char_ngrams(text, 2)]
    features += ["C3_" + f for f in char_ngrams(text, 3)]
    features += ["C4_" + f for f in char_ngrams(text, 4)]
    return features


In [6]:
def build_vocabulary(texts, min_df=2, max_features=60000):
    N = len(texts)
    doc_freq = Counter()
    for text in texts:
        doc_freq.update(set(get_features(text)))

    filtered = [(f, c) for f, c in doc_freq.items() if c >= min_df]
    filtered.sort(key=lambda x: x[1], reverse=True)
    filtered = filtered[:max_features]

    vocabulary = {feat: i for i, (feat, _) in enumerate(filtered)}
    doc_freq_arr = np.zeros(len(vocabulary), dtype=np.float64)
    for feat, c in filtered:
        doc_freq_arr[vocabulary[feat]] = c
    return vocabulary, doc_freq_arr, N


t0 = time.time()
vocabulary, doc_freq_arr, N_train = build_vocabulary(train_df["text"].tolist(), min_df=2, max_features=60000)
print(f"Vocabulary built in {time.time()-t0:.1f}s -> size: {len(vocabulary)}")


Vocabulary built in 6.7s -> size: 60000


In [7]:
def idf_standard(doc_freq_arr, N):
    """Assignment 1's smoothed IDF (unnormalized baseline)."""
    return np.log((1.0 + N) / (1.0 + doc_freq_arr)) + 1.0


def idf_max_df_normalized(doc_freq_arr):
    """IDF normalized relative to the most frequent term across the training corpus.
    max_df = document frequency of the single most common n-gram in the vocabulary.
    Terms as common as the most frequent term get idf ~ 0; rarer terms get progressively
    larger idf, scaled relative to how dominant the most frequent term is.
    """
    max_df = doc_freq_arr.max()
    return np.log((1.0 + max_df) / (1.0 + doc_freq_arr))


idf_std = idf_standard(doc_freq_arr, N_train)
idf_maxdf = idf_max_df_normalized(doc_freq_arr)

print("Standard IDF   -> min: {:.4f}, max: {:.4f}, mean: {:.4f}".format(idf_std.min(), idf_std.max(), idf_std.mean()))
print("Max-df norm IDF-> min: {:.4f}, max: {:.4f}, mean: {:.4f}".format(idf_maxdf.min(), idf_maxdf.max(), idf_maxdf.mean()))


Standard IDF   -> min: 2.5453, max: 8.2552, mean: 7.3366
Max-df norm IDF-> min: 0.0000, max: 5.7099, mean: 4.7913


In [8]:
def build_count_matrix(df, vocabulary, verbose_every=5000):
    n = len(df)
    rows_idx, cols_idx, data = [], [], []
    texts = df["text"].tolist()
    for r, text in enumerate(texts):
        feats = get_features(text)
        counts = Counter(f for f in feats if f in vocabulary)
        for f, c in counts.items():
            rows_idx.append(r)
            cols_idx.append(vocabulary[f])
            data.append(float(c))
        if verbose_every and (r + 1) % verbose_every == 0:
            print(f"  counted {r + 1}/{n}")
    return sp.csr_matrix((data, (rows_idx, cols_idx)), shape=(n, len(vocabulary)))


print("Building raw count matrices...")
C_train = build_count_matrix(train_df, vocabulary)
C_val   = build_count_matrix(val_df, vocabulary)
C_test  = build_count_matrix(test_df, vocabulary)
print(C_train.shape, C_val.shape, C_test.shape)

y_train = np.array([label2idx[l] for l in train_df["label"]])
y_val   = np.array([label2idx[l] for l in val_df["label"]])
y_test  = np.array([label2idx[l] for l in test_df["label"]])


Building raw count matrices...
  counted 5000/18400
  counted 10000/18400
  counted 15000/18400
(18400, 60000) (2300, 60000) (2300, 60000)


In [9]:
def row_totals(counts):
    tot = np.asarray(counts.sum(axis=1)).flatten()
    tot[tot == 0] = 1.0
    return tot


def row_maxes(counts):
    m = counts.max(axis=1)
    m = np.asarray(m.todense()).flatten() if sp.issparse(m) else np.asarray(m).flatten()
    m[m == 0] = 1.0
    return m


def apply_tf(counts, mode):
    if mode == "raw":
        return counts.copy()
    elif mode == "length":
        tot = row_totals(counts)
        return counts.multiply(1.0 / tot[:, None]).tocsr()
    elif mode == "max":
        mx = row_maxes(counts)
        return counts.multiply(1.0 / mx[:, None]).tocsr()
    else:
        raise ValueError(f"unknown tf mode: {mode}")


def apply_idf(tf_matrix, idf_vec):
    return tf_matrix.multiply(idf_vec[None, :]).tocsr()


def l2_normalize_rows(matrix):
    sq_sum = np.asarray(matrix.multiply(matrix).sum(axis=1)).flatten()
    norms = np.sqrt(sq_sum)
    norms[norms == 0] = 1.0
    return matrix.multiply(1.0 / norms[:, None]).tocsr()


def build_tfidf(counts, tf_mode, idf_vec):
    tf = apply_tf(counts, tf_mode)
    tfidf = apply_idf(tf, idf_vec)
    tfidf = l2_normalize_rows(tfidf)
    return tfidf


TF_MODES = ["raw", "length", "max"]
IDF_MODES = [("standard", idf_std), ("max_df", idf_maxdf)]

combinations = [(tf_mode, idf_name, idf_vec) for tf_mode in TF_MODES for idf_name, idf_vec in IDF_MODES]
print(f"Total combinations: {len(combinations)}")
for tf_mode, idf_name, _ in combinations:
    print(f"  - TF={tf_mode:7s} x IDF={idf_name}")


Total combinations: 6
  - TF=raw     x IDF=standard
  - TF=raw     x IDF=max_df
  - TF=length  x IDF=standard
  - TF=length  x IDF=max_df
  - TF=max     x IDF=standard
  - TF=max     x IDF=max_df


In [10]:
vectorized = {}
for tf_mode, idf_name, idf_vec in combinations:
    Xtr = build_tfidf(C_train, tf_mode, idf_vec)
    Xva = build_tfidf(C_val, tf_mode, idf_vec)
    Xte = build_tfidf(C_test, tf_mode, idf_vec)
    vectorized[(tf_mode, idf_name)] = (Xtr, Xva, Xte)
    print(f"TF={tf_mode:7s} IDF={idf_name:8s} -> train {Xtr.shape}, nnz={Xtr.nnz}")


TF=raw     IDF=standard -> train (18400, 60000), nnz=3089674
TF=raw     IDF=max_df   -> train (18400, 60000), nnz=3089674
TF=length  IDF=standard -> train (18400, 60000), nnz=3089674
TF=length  IDF=max_df   -> train (18400, 60000), nnz=3089674
TF=max     IDF=standard -> train (18400, 60000), nnz=3089674
TF=max     IDF=max_df   -> train (18400, 60000), nnz=3089674


In [11]:
class SoftmaxRegressionScratch:
    def __init__(self, num_features, num_classes, learning_rate=0.5,
                 epochs=25, batch_size=64, l2_reg=1e-4, seed=42):
        rng = np.random.default_rng(seed)
        self.W = rng.normal(0, 0.01, size=(num_classes, num_features))
        self.b = np.zeros(num_classes)
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.l2 = l2_reg
        self.num_classes = num_classes
        self.history = {"train_loss": []}

    @staticmethod
    def softmax(Z):
        Z = Z - Z.max(axis=1, keepdims=True)
        E = np.exp(Z)
        return E / E.sum(axis=1, keepdims=True)

    def fit(self, X, y, verbose=False):
        n = X.shape[0]
        Y = np.zeros((n, self.num_classes))
        Y[np.arange(n), y] = 1.0
        indices = np.arange(n)

        for epoch in range(self.epochs):
            rng = np.random.default_rng(1000 + epoch)
            rng.shuffle(indices)
            total_loss = 0.0
            for start in range(0, n, self.batch_size):
                b_idx = indices[start:start + self.batch_size]
                Xb = X[b_idx]
                Yb = Y[b_idx]
                yb = y[b_idx]
                bs = Xb.shape[0]

                Zb = Xb.dot(self.W.T) + self.b
                Pb = self.softmax(np.asarray(Zb))
                dZ = (Pb - Yb) / bs
                grad_W = np.asarray(Xb.T.dot(dZ)).T + self.l2 * self.W
                grad_b = dZ.sum(axis=0)

                self.W -= self.lr * grad_W
                self.b -= self.lr * grad_b

                logp = np.log(np.clip(Pb[np.arange(bs), yb], 1e-12, None))
                total_loss += -logp.sum()

            self.history["train_loss"].append(total_loss / n)
            if verbose:
                print(f"  epoch {epoch+1}/{self.epochs} loss={self.history['train_loss'][-1]:.4f}")

    def predict_proba(self, X):
        Z = X.dot(self.W.T) + self.b
        return self.softmax(np.asarray(Z))

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)


In [12]:
def confusion_counts(y_true, y_pred, num_classes):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    tp = np.zeros(num_classes); fp = np.zeros(num_classes); fn = np.zeros(num_classes)
    for c in range(num_classes):
        tp[c] = np.sum((y_pred == c) & (y_true == c))
        fp[c] = np.sum((y_pred == c) & (y_true != c))
        fn[c] = np.sum((y_pred != c) & (y_true == c))
    return tp, fp, fn


def precision_recall_f1_per_class(y_true, y_pred, num_classes):
    tp, fp, fn = confusion_counts(y_true, y_pred, num_classes)
    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    recall = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    denom = precision + recall
    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(tp), where=denom > 0)
    return precision, recall, f1


def macro_f1_score(y_true, y_pred, num_classes):
    _, _, f1 = precision_recall_f1_per_class(y_true, y_pred, num_classes)
    return float(np.mean(f1))


def accuracy_score_custom(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


In [13]:
results = []
trained_models = {}

for (tf_mode, idf_name), (Xtr, Xva, Xte) in vectorized.items():
    print(f"\n=== TF={tf_mode} | IDF={idf_name} ===")
    t0 = time.time()
    clf = SoftmaxRegressionScratch(
        num_features=Xtr.shape[1],
        num_classes=len(languages),
        learning_rate=0.7,
        epochs=25,
        batch_size=64,
        l2_reg=1e-4,
        seed=RANDOM_SEED,
    )
    clf.fit(Xtr, y_train, verbose=False)
    train_time = time.time() - t0

    val_preds = clf.predict(Xva)
    test_preds = clf.predict(Xte)

    val_acc = accuracy_score_custom(y_val, val_preds)
    val_f1 = macro_f1_score(y_val, val_preds, len(languages))
    test_acc = accuracy_score_custom(y_test, test_preds)
    test_f1 = macro_f1_score(y_test, test_preds, len(languages))

    print(f"  train_time={train_time:.1f}s  val_acc={val_acc:.4f}  val_macroF1={val_f1:.4f}  "
          f"test_acc={test_acc:.4f}  test_macroF1={test_f1:.4f}")

    trained_models[(tf_mode, idf_name)] = clf
    results.append({
        "tf_mode": tf_mode,
        "idf_mode": idf_name,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1,
        "test_accuracy": test_acc,
        "test_macro_f1": test_f1,
        "train_time_s": train_time,
    })



=== TF=raw | IDF=standard ===
  train_time=155.1s  val_acc=0.9391  val_macroF1=0.9382  test_acc=0.9291  test_macroF1=0.9281

=== TF=raw | IDF=max_df ===
  train_time=154.2s  val_acc=0.9426  val_macroF1=0.9418  test_acc=0.9391  test_macroF1=0.9383

=== TF=length | IDF=standard ===
  train_time=147.2s  val_acc=0.9391  val_macroF1=0.9382  test_acc=0.9291  test_macroF1=0.9281

=== TF=length | IDF=max_df ===
  train_time=165.4s  val_acc=0.9426  val_macroF1=0.9418  test_acc=0.9391  test_macroF1=0.9383

=== TF=max | IDF=standard ===
  train_time=155.7s  val_acc=0.9391  val_macroF1=0.9382  test_acc=0.9291  test_macroF1=0.9281

=== TF=max | IDF=max_df ===
  train_time=150.5s  val_acc=0.9426  val_macroF1=0.9418  test_acc=0.9391  test_macroF1=0.9383


## 10. Comparison table — normalized vs. unnormalized TF/IDF

In [14]:
results_df = pd.DataFrame(results)
results_df["combination"] = results_df["tf_mode"] + " TF  x  " + results_df["idf_mode"] + " IDF"
results_df = results_df.sort_values("test_macro_f1", ascending=False).reset_index(drop=True)

display_cols = ["combination", "tf_mode", "idf_mode", "val_accuracy", "val_macro_f1", "test_accuracy", "test_macro_f1", "train_time_s"]
results_df[display_cols]


,combination,tf_mode,idf_mode,val_accuracy,val_macro_f1,test_accuracy,test_macro_f1,train_time_s
0,raw TF x max_df IDF,raw,max_df,0.942609,0.941781,0.93913,0.938260,154.210259
1,length TF x max_df IDF,length,max_df,0.942609,0.941781,0.93913,0.938260,165.420261
2,max TF x max_df IDF,max,max_df,0.942609,0.941781,0.93913,0.938260,150.482318
3,raw TF x standard IDF,raw,standard,0.939130,0.938186,0.92913,0.928091,155.069760
4,length TF x standard IDF,length,standard,0.939130,0.938186,0.92913,0.928091,147.244829
5,max TF x standard IDF,max,standard,0.939130,0.938186,0.92913,0.928091,155.707519


In [15]:
best = results_df.iloc[0]
worst = results_df.iloc[-1]
print(f"Best combination : {best['combination']}  -> test macro-F1 = {best['test_macro_f1']:.4f}")
print(f"Worst combination: {worst['combination']}  -> test macro-F1 = {worst['test_macro_f1']:.4f}")

print("\nEffect of TF normalization (averaged across both IDF modes):")
print(results_df.groupby("tf_mode")[["val_macro_f1", "test_macro_f1"]].mean())

print("\nEffect of IDF normalization (averaged across all three TF modes):")
print(results_df.groupby("idf_mode")[["val_macro_f1", "test_macro_f1"]].mean())


Best combination : raw TF  x  max_df IDF  -> test macro-F1 = 0.9383
Worst combination: max TF  x  standard IDF  -> test macro-F1 = 0.9281

Effect of TF normalization (averaged across both IDF modes):
         val_macro_f1  test_macro_f1
tf_mode                             
length       0.939983       0.933175
max          0.939983       0.933175
raw          0.939983       0.933175

Effect of IDF normalization (averaged across all three TF modes):
          val_macro_f1  test_macro_f1
idf_mode                             
max_df        0.941781       0.938260
standard      0.938186       0.928091
